# OCR de Kanjis N2-N5 com YOLOv8

Este notebook foi projetado para rodar no **Kaggle** ou **Google Colab** (com GPU ativada).

Ele orquestra o pipeline completo do projeto:
1. Instala dependências
2. Clona o repositório
3. Baixa fontes CJK
4. Gera o dataset sintético balanceado (~69k imagens)
5. Treina o YOLOv8n por 50 épocas
6. Disponibiliza os pesos treinados para download
7. Executa inferência em imagens de teste

## 0. Verificar GPU

In [ ]:
import torch
print(f'CUDA disponível: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('AVISO: GPU não encontrada. Treinamento será muito mais lento.')

## 1. Instalar Dependências e Clonar Repositório

In [ ]:
!pip install -q ultralytics opencv-python-headless Pillow tqdm

import os
REPO_NAME = 'OCR-de-kanjis-N2-N5-com-YoloV8'
GITHUB_URL = 'https://github.com/Thomaz332/OCR-de-kanjis-N2-N5-com-YoloV8.git'

if os.path.exists(REPO_NAME):
    print(f'Repositório {REPO_NAME} já existe. Atualizando...')
    os.system(f'cd {REPO_NAME} && git pull')
else:
    os.system(f'git clone {GITHUB_URL}')

%cd {REPO_NAME}
print('Diretório atual:', os.getcwd())

## 1.5 Carregar Configuração do Pipeline

As duas variáveis abaixo (caminho do dataset sintético anexado e caminho do
checkpoint anexado) são lidas de `kaggle_config.json`, na raiz do repositório.
Esse arquivo é reescrito automaticamente por `scripts/kaggle_pipeline.py` antes
de cada `kaggle kernels push` — não há necessidade de editar nada manualmente
aqui ou na interface do Kaggle.

In [ ]:
# === PIPELINE_CONFIG ===
# Este dicionário é reescrito diretamente nesta célula por
# scripts/kaggle_pipeline.py (a partir de kaggle_config.json) logo antes
# de cada `kaggle kernels push` -- não há leitura de arquivo em tempo de
# execução, então não há dependência de nada além do próprio notebook.
# Rodando manualmente (Save & Run All) sem passar pelo pipeline, os
# valores abaixo (None) fazem o notebook gerar o dataset e treinar do
# zero, como no comportamento original.
PIPELINE_CONFIG = {
    'synthetic_dataset_path': '/kaggle/input/ic-ocr-synthetic-dataset',
    'checkpoint_path': '/kaggle/input/ic-ocr-checkpoint',
}

print('PIPELINE_CONFIG:', PIPELINE_CONFIG)

## 2. Baixar Fontes CJK

In [ ]:
!python src/data/download_fonts.py

import glob
fontes = glob.glob('assets/fonts/*')
print(f'Fontes disponíveis: {[f.split("/")[-1] for f in fontes]}')

## 3. Gerar Dataset Sintético N2-N5

**Estratégia:** Dataset 100% sintético e balanceado — cada kanji recebe o mesmo número de
imagens, eliminando o viés de frequência natural dos mangás.

- ~690 classes (689 kanjis N2-N5 + UNKNOWN_N1)
- 100 imagens/classe = ~69.000 imagens no total
- Split automático 80/20 (treino/val)
- Augmentations: distorção elástica, screentone, textura de papel, etc.

In [ ]:
import os

synthetic_dir = 'data/synthetic'
cached_dir = PIPELINE_CONFIG.get('synthetic_dataset_path')
cached_yaml = os.path.join(cached_dir, 'data.yaml') if cached_dir else None

if cached_dir and os.path.exists(cached_yaml):
    print(f'Dataset sintético anexado encontrado em: {cached_dir}')
    print('Reaproveitando (sem reprocessar imagens) — apenas ajustando data.yaml local...')
    os.makedirs(synthetic_dir, exist_ok=True)
    with open(cached_yaml, 'r', encoding='utf-8') as f:
        yaml_lines = f.readlines()
    with open(os.path.join(synthetic_dir, 'data.yaml'), 'w', encoding='utf-8') as f:
        for line in yaml_lines:
            f.write(f'path: {cached_dir}\n' if line.startswith('path:') else line)
    train_dir = os.path.join(cached_dir, 'images', 'train')
    val_dir = os.path.join(cached_dir, 'images', 'val')
else:
    if cached_dir:
        print(f'AVISO: synthetic_dataset_path={cached_dir!r} não contém data.yaml válido. Gerando do zero...')
    else:
        print('Nenhum dataset sintético anexado. Gerando do zero (processo demorado)...')
    get_ipython().system('python src/data/generate_synthetic_images.py')
    train_dir = os.path.join(synthetic_dir, 'images', 'train')
    val_dir = os.path.join(synthetic_dir, 'images', 'val')

train_imgs = len(os.listdir(train_dir))
val_imgs = len(os.listdir(val_dir))
print(f'Imagens de treino: {train_imgs}')
print(f'Imagens de val:    {val_imgs}')
print(f'Total:             {train_imgs + val_imgs}')

## 4. Treinar o Modelo YOLOv8n

In [ ]:
import os
import shutil
from ultralytics import YOLO

data_path = os.path.abspath('data/synthetic/data.yaml')
print(f'Usando configuração: {data_path}')

checkpoint_dir = PIPELINE_CONFIG.get('checkpoint_path')
local_run_dir = os.path.join('yolo_kanji', 'n2_n5_model')
checkpoint_last_pt = os.path.join(checkpoint_dir, 'weights', 'last.pt') if checkpoint_dir else None
has_checkpoint = bool(checkpoint_dir) and os.path.exists(checkpoint_last_pt)
resumed = False

if has_checkpoint:
    print(f'Checkpoint anexado encontrado em: {checkpoint_dir}')
    print('Copiando estado do treino anterior para continuar de onde parou...')
    os.makedirs(os.path.dirname(local_run_dir), exist_ok=True)
    if os.path.exists(local_run_dir):
        shutil.rmtree(local_run_dir)
    shutil.copytree(checkpoint_dir, local_run_dir)
    local_last_pt = os.path.join(local_run_dir, 'weights', 'last.pt')
    try:
        model = YOLO(local_last_pt)
        results = model.train(resume=True)
        resumed = True
    except Exception as e:
        print(f'Não foi possível retomar o treino original (motivo: {e}).')
        print('Continuando como fine-tune a partir dos pesos do checkpoint (novo ciclo de épocas).')
        model = YOLO(local_last_pt)
else:
    if checkpoint_dir:
        print(f'AVISO: checkpoint_path={checkpoint_dir!r} não contém weights/last.pt. Treinando do zero.')
    else:
        print('Nenhum checkpoint anexado. Treinando do zero (yolov8n.pt).')
    model = YOLO('yolov8n.pt')

if not resumed:
    results = model.train(
        data=data_path,
        epochs=50,
        imgsz=640,
        batch=16,
        device=0,       # GPU
        project='yolo_kanji',
        name='n2_n5_model',
        exist_ok=True,
        plots=True,
        verbose=True,
    )

print('Treinamento concluído!')

## 5. Download dos Pesos Treinados

In [ ]:
from IPython.display import FileLink, display
import glob, os

train_dirs = glob.glob('yolo_kanji/n2_n5_model*')
if not train_dirs:
    train_dirs = glob.glob('runs/detect/train*')

if train_dirs:
    latest = max(train_dirs, key=os.path.getmtime)
    weights = f'{latest}/weights/best.pt'
    if os.path.exists(weights):
        print(f'Modelo treinado: {weights}')
        print(f'Tamanho: {os.path.getsize(weights) / 1e6:.1f} MB')
        display(FileLink(weights))
    else:
        print('Pesos ainda não gerados.')
else:
    print('Nenhum treinamento encontrado.')

## 6. Testar o Modelo em Imagens de Mangá

Faça upload de imagens de mangá na pasta `manga_test/` e rode a inferência.

In [ ]:
from ultralytics import YOLO
import glob, os, cv2
import matplotlib.pyplot as plt
from IPython.display import FileLink, display

test_folder = 'manga_test/'
os.makedirs(test_folder, exist_ok=True)

imagens = glob.glob(f'{test_folder}/*.jpg') + glob.glob(f'{test_folder}/*.png')

if not imagens:
    print(f'Nenhuma imagem em {test_folder}.')
    print('Faça upload de imagens de mangá nesta pasta e rode a célula novamente.')
else:
    print(f'{len(imagens)} imagem(ns) encontrada(s).')

    # Carregar modelo treinado
    train_dirs = glob.glob('yolo_kanji/n2_n5_model*') or glob.glob('runs/detect/train*')
    latest = max(train_dirs, key=os.path.getmtime)
    model = YOLO(f'{latest}/weights/best.pt')

    # Inferência
    results = model(test_folder, conf=0.25, save=True, project='resultados_manga', name='predicoes')

    # Exibir primeiro resultado
    pred_imgs = glob.glob('resultados_manga/predicoes/*.jpg')
    if pred_imgs:
        img = cv2.cvtColor(cv2.imread(pred_imgs[0]), cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(12, 12))
        plt.imshow(img)
        plt.axis('off')
        plt.title('Detecções N2-N5 (primeiro resultado)')
        plt.show()

    # Zipar resultados
    os.system('zip -q -r resultados_manga.zip resultados_manga/')
    print('Download dos resultados:')
    display(FileLink('resultados_manga.zip'))